# CDT ML for Time Series: EHR tokenization

This lab introduces the use of EHR tokenization. In the previous labs, you would have worked with clinical measurements and machine learning. Here, you learn to use the preprocessed data for a different purpose: converting patient histories into tokens for sequence models. First, you understand the tokenization by using four features and 12 time steps to explore Triplet, EIC, and Text-Code representations. Second, you apply tokenization to Mortality Prediction by turning them into embeddings and train a Transformer to predict the mortality label.

We will try three versions of tokenization ideas: Triplet, Event–Interval–Code (EIC), and Text-Code representations. You will build tokens, compare sequence length and vocabulary, check what can be reconstructed, and explain the trade-offs.

Main Contact: Dr. Patitapaban Palo (pati.palo@eng.ox.ac.uk)

Before you start: download and extract the data(zip) from the course materials.

## 0: Environment Setup and Loading Packages

In this section, you shall set up the Python environment and import the libraries needed for this tutorial.

**If you are using Colab, you can skip the virtual environment setup and go directly to Section 1.**

Open a terminal and run:

conda create --name ehrTokenizationEnv python=3.11 -y
conda activate ehrTokenizationEnv

**Prerequisites**
numpy pandas matplotlib ipykernel scikit-learn torch

In [ ]:
!pip install numpy pandas matplotlib ipykernel scikit-learn torch

## 1. Load and prepare data

We use the same ICU dataset as in the earlier tutorial on EHR data preprocessing by Yujiang Wang. Expected data folder structure:

```text
data/
    X_data.npy
    y_data.npy
    feature_names.txt
```
The three files:
- **`X_data.npy`**: Contains the clinical measurements for 9,622 patients, with 48 time steps and 59 features per patient.
- **`y_data.npy`**: Contains the corresponding mortality labels for each patient (`0` = survived, `1` = died).
- **`feature_names.txt`**: Contains the names of the 59 clinical features, such as heart rate, oxygen saturation, temperature, etc.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Please update the path for the data_dir.
data_dir = Path('data')
if not (data_dir / 'X_data.npy').is_file() and (data_dir / 'data' / 'X_data.npy').is_file():
    data_dir = data_dir / 'data'  # Handles a folder extracted with an extra data/ level.

if not (data_dir / 'X_data.npy').is_file():
    raise FileNotFoundError(
        'Could not find data/X_data.npy. Extract data-1.zip first and put '
        'the data folder beside this notebook (or upload the folder to Colab).'
    )

X = np.load(data_dir / 'X_data.npy')
y = np.load(data_dir / "y_data.npy").reshape(-1)
feature_names = (data_dir / 'feature_names.txt').read_text(encoding='utf-8').splitlines()[1:]

# Check the shapes
print("X shape:", X.shape)
print("y shape:", y.shape)

assert len(X) == len(y)

# To understand tokenization, we use just four features and twelve time steps.
chosen = ['Heart Rate', 'Oxygen saturation', 'Respiratory rate', 'Temperature']
indices = [feature_names.index(name) for name in chosen]

# These first 500 patients for understanding value thresholds.
reference = X[:500, :12, :][:, :, indices].astype(float)
patient = X[500, :12, :][:, indices].astype(float)

# This is to check and avoid silently assigning a missing value to the HIGH category.
if not np.isfinite(reference).all() or not np.isfinite(patient).all():
    raise ValueError('Selected measurements contain missing or invalid values. Check the prepared data.')

print('Full array:', X.shape)
print('Patient example:', patient.shape)
print('Features:', chosen)
assert patient.shape == (12, 4)

### A shared set of value categories
We first map each feature to `LOW`, `MID`, or `HIGH` using the 33rd and 67th percentiles of the reference data. These categories are relative to this dataset, not medical diagnoses.

Using the same categories across all three strategies makes the comparison easier to interpret.The categories are based on the reference measurements for this small demonstration. They are not clinical cut-offs.

In [ ]:
edges = np.quantile(reference, [1/3, 2/3], axis=(0, 1))

def category(value, feature_index):
    if value < edges[0, feature_index]:
        return 'LOW'
    if value < edges[1, feature_index]:
        return 'MID'
    return 'HIGH'

print('Heart rate categories:', [category(patient[t, 0], 0) for t in range(5)])

### 2. Understanding Triplet representation
A triplet keeps when, what, and which value category. For example: `(2, 'Heart Rate', 'HIGH')`.

Your task: complete `make_triplets`. Add one tuple for each time step and feature. Start with an empty list, then use two loops.

Check: 12 time steps × 4 features = 48 triplets.

In [ ]:
def make_triplets(values, names):
    tokens = []
    # Hint: loop over time steps and features, and append (time_index, feature_name, category) tuples.
    raise NotImplementedError('Complete make_triplets')

triplets = make_triplets(patient, chosen)
print('First 6:', triplets[:6])
assert len(triplets) == 48
assert triplets[0][0] == 0 and triplets[0][1] == 'Heart Rate'

**Question:** The original numerical value is gone. Could we recover it exactly from `LOW`, `MID`, or `HIGH`? Why not?

### 3. Understanding EIC representation
In many EHRs, events happen at irregular times. We can represent a sequence using event codes and the gaps between consecutive events.

Our array is on a regular grid, so we create a teaching example: for each feature, keep an event only when its category changes. We then add a gap token such as `GAP_3` between retained events. `GAP_3` means three time-step indices, not three hours.

Your task: complete `make_eic`. For each feature, keep the first category, then only category changes. Insert a gap token before each later event.

In [ ]:
def make_eic(values, names):
    tokens = []
    # Hint: for each feature, find the times when its category changes, append a feature marker, then the first event token, and GAP_<difference in indices>.
    raise NotImplementedError('Complete make_eic')

eic_tokens = make_eic(patient, chosen)
print('First 16:', eic_tokens[:16])
assert eic_tokens.count('FEATURE:Heart Rate') == 1
assert any(token.startswith('GAP_') for token in eic_tokens)

**Question:** Why can this sequence be shorter than the triplets? What assumption do we make when we omit repeated categories? What would happen if a real clinical measurement were missing rather than unchanged?

### 4. Understanding Text-Code representation
Another approach is to serialize each event as a text token, for example `t2|Heart_Rate|HIGH`.

This is not natural-language tokenization or a model-specific text tokenizer. It simply shows how a structured event can become a string.

Your task: make one string per time step and feature, keeping the time index and category. Use underscores instead of spaces in feature names.

In [ ]:
def make_text_codes(values, names):
    tokens = []
    # Hint: build tokens like 't2|Heart_Rate|HIGH'.
    raise NotImplementedError('Complete make_text_codes')

text_tokens = make_text_codes(patient, chosen)
print('First 6:', text_tokens[:6])
assert len(text_tokens) == 48
assert text_tokens[0].startswith('t0|Heart_Rate|')

**Question:** Here, the time index is part of each string. If every possible time and value were written into separate tokens, what might happen to vocabulary size?

### 5. Compare representations
We now compare token count, vocabulary size, and information retained. Note: a triplet is one structured item, while the EIC example uses separate gap and value tokens. These counts are useful for discussion.

For vocabulary, count distinct symbols after converting structured triplets into components.
Your task: complete `token_symbols` so triplets contribute three symbols (`T_...`, `FEATURE_...`, `VALUE_...`), while string tokens contribute one symbol each.

In [ ]:
def token_symbols(items, style):
    # Hint: if style == 'triplet', split each tuple into 3 symbol strings, otherwise return the list of string tokens unchanged.
    raise NotImplementedError('Complete token_symbols')

styles = {'Triplet': 'triplet', 'EIC': 'eic', 'Text-Code': 'text'}
makers = {'Triplet': make_triplets, 'EIC': make_eic, 'Text-Code': make_text_codes}
rows = []
for label, style in styles.items():
    sequences = [makers[label](X[i, :12, :][:, indices], chosen) for i in range(500, 520)]
    symbols = [s for seq in sequences for s in token_symbols(seq, style)]
    rows.append({'strategy': label,
                 'mean_items_per_patient': np.mean([len(seq) for seq in sequences]),
                 'unique_symbols': len(set(symbols))})
summary = pd.DataFrame(rows)
display(summary)
assert len(summary) == 3

### Analyze: What information have we lost?
All three representations use categories, not the original numbers. We can estimate how much numerical detail was lost by replacing each category with the median reference value for that feature and category.

Your task: write `reconstruct` to replace each original measurement with the median of its category. Then calculate mean absolute error (MAE). This is a test of value discretization, not a test of which sequence model is better.

In [ ]:
category_names = ['LOW', 'MID', 'HIGH']
centres = np.zeros((3, len(chosen)))
for j in range(len(chosen)):
    vals = reference[:, :, j].ravel()
    for k, label in enumerate(category_names):
        selected = np.array([category(v, j) == label for v in vals])
        centres[k, j] = np.median(vals[selected]) if selected.any() else np.median(vals)

def reconstruct(values):
    # Hint: return an array of the same shape as values. For each entry, use the centre of its feature's LOW/MID/HIGH category.
    raise NotImplementedError('Complete reconstruct')

recovered = reconstruct(patient)
mae = np.mean(np.abs(patient - recovered))
print(f'Average absolute reconstruction error: {mae:.3f} (processed-value units)')
assert recovered.shape == patient.shape
assert np.isfinite(mae)

### Comparison plot
The plot compares the number of representation items.

In [ ]:
ax = summary.plot.bar(x='strategy', y='mean_items_per_patient', legend=False, rot=15)
ax.set_ylabel('Average items per patient')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## Part 2 - Applying Tokenization to Mortality Prediction using transformer

We now use every patient and every time step, and prepare the full patient sequence. A direct event-level Transformer would receive 48 × 59 = 2,832 measurement tokens per patient. That would be slow because attention compares token pairs.

Instead, we group the 59 feature representations at each time step into one time-step vector. The Transformer therefore sees 48 tokens per patient, while each token still uses all 59 features.

For the full experiment:
- Triplet will use continuous values, feature embeddings, and time embeddings.
- EIC will use three training set-based value bins per feature, plus an interval embedding. Since the grid is regular, the interval is constant.
- Text Code will use 16 training set-based value bins per feature, giving a finer discrete representation. The classroom string example above uses rounded values; the full model uses bins to avoid an enormous vocabulary.

The categories are computed using training data only. This avoids using information from the test set during tokenization.

In [ ]:
import io, zipfile, time, random, os
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

all_idx = np.arange(len(y))
train_idx, temp_idx = train_test_split(all_idx, test_size=0.30, random_state=SEED, stratify=y)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED, stratify=y[temp_idx])
print('Patients — train:', len(train_idx), 'validation:', len(val_idx), 'test:', len(test_idx))

# Robust scaling using training data
median = np.median(X[train_idx], axis=(0, 1))
q25, q75 = np.quantile(X[train_idx], [0.25, 0.75], axis=(0, 1))
scale = np.maximum(q75 - q25, 1.0)
X_scaled = np.clip((X - median) / scale, -8, 8).astype(np.float32)

# Bin thresholds are fitted using training data.
def make_bins(n_bins):
    thresholds = np.quantile(X_scaled[train_idx], np.arange(1, n_bins)/n_bins, axis=(0, 1))
    # Output shape: patients x time x features. Duplicate quantiles are allowed.
    return (X_scaled[..., None] > thresholds.T[None, None, :, :]).sum(axis=-1).astype(np.int64)

BINS_EIC = make_bins(3)
BINS_TEXT = make_bins(16)
assert X_scaled.shape == BINS_EIC.shape == BINS_TEXT.shape == (9622, 48, 59)
print('All 59 features and 48 time steps retained.')

## 6. Build the input embeddings 

A Transformer expects an input with shape `(batch, time, d_model)`. We set `d_model=32`.

Your tasks: complete three embedding modules. For Triplet, combine the feature identity and numerical value for each measurement, then combine the 59 features with feature-specific value weights. For EIC and Text Code, embed the feature-value categories and average across features. Add a time-position embedding to each time step.

We average over features, not over time. The Transformer must still see the sequence of 48 time steps.

In [ ]:
D_MODEL = 32
N_FEATURES = X.shape[2]
N_TIMES = X.shape[1]

class TripletInput(nn.Module):
    def __init__(self, d_model=32):
        super().__init__()
        self.feature = nn.Embedding(N_FEATURES, d_model)
        self.value_weights = nn.Parameter(torch.randn(N_FEATURES, d_model) * 0.02)
        self.time = nn.Embedding(N_TIMES, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # Hint: Combine feature embeddings with feature-specific weighted values, sum and scale by sqrt(N_FEATURES), 
        # then add time embeddings and apply LayerNorm. Expected output shape: (batch, 48, 32)
        raise NotImplementedError

class DiscreteInput(nn.Module):
    def __init__(self, n_bins, d_model=32):
        super().__init__()
        self.n_bins = n_bins
        self.feature_value = nn.Embedding(N_FEATURES * n_bins, d_model)
        self.time = nn.Embedding(N_TIMES, d_model)

    def forward(self, bins):
        # Hint: Create unique token IDs from feature and bin IDs, embed and average across features, then add time embeddings.
        # Expected output shape: (batch, 48, 32)
        raise NotImplementedError

# EIC adds a time-gap embedding to the category representation.
class EICInput(DiscreteInput):
    def __init__(self, d_model=32):
        super().__init__(3, d_model)
        self.interval = nn.Embedding(2, d_model)  # 0=first step, 1=one step later

    def forward(self, bins):
        x = super().forward(bins)
        gaps = torch.ones(bins.shape[1], dtype=torch.long, device=bins.device)
        gaps[0] = 0
        return x + self.interval(gaps)[None, :, :]

# These shape checks should pass for the next steps.
inputs = [torch.from_numpy(X_scaled[:2]), torch.from_numpy(BINS_EIC[:2]), torch.from_numpy(BINS_TEXT[:2])]
assert TripletInput()(inputs[0]).shape == (2, 48, 32)
assert EICInput()(inputs[1]).shape == (2, 48, 32)
assert DiscreteInput(16)(inputs[2]).shape == (2, 48, 32)
print('Embedding shape checks passed.')

## 7. Build the Transformer

We use one encoder layer, two attention heads, and a classifier. The model should produce one mortality logit per patient.

Your task: finish the forward pass: make the embeddings, apply the Transformer, pool across the 48 time steps, and calculate the logit.

In [ ]:
class PatientTransformer(nn.Module):
    def __init__(self, representation, d_model=32):
        super().__init__()
        self.representation = representation
        self.input_layer = {
            'triplet': lambda: TripletInput(d_model),
            'eic': lambda: EICInput(d_model),
            'text_code': lambda: DiscreteInput(16, d_model),
        }[representation]()
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=2,
            dim_feedforward=64, dropout=0.1, batch_first=True, norm_first=False)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, inputs):
        # Hint: Pass inputs through the input layer and Transformer, average over time, apply LayerNorm & classifier to return the logit.
        raise NotImplementedError

for representation, batch in zip(['triplet','eic','text_code'], inputs):
    model = PatientTransformer(representation)
    assert model(batch).shape == (2,)
print('Transformer shape checks passed.')

## 8. Train and compare

The training loop is provided. It uses all training data, validates each epoch, and evaluates the held-out test group once, after choosing the best epoch by validation AUROC.

Train Triplet, EIC, and Text Code with the same data split and training setup. The default is 10 epochs; increase to 15–20 after class if learning is still improving. Results will vary with the random seed.

Your Task: Complete the training loop in train_model().

In [ ]:
from copy import deepcopy
from sklearn.metrics import roc_auc_score, average_precision_score

# Full dataset is used in all three experiments.
REPRESENTATIONS = ['triplet', 'eic', 'text_code']
DATA = {
    'triplet': X_scaled,
    'eic': BINS_EIC,
    'text_code': BINS_TEXT,
}

def loaders_for(representation, batch_size=128):
    arr = DATA[representation]
    dtype = torch.float32 if representation in ('triplet',) else torch.long
    def loader(ids, shuffle=False):
        ds = TensorDataset(torch.as_tensor(arr[ids], dtype=dtype),
                           torch.as_tensor(y[ids], dtype=torch.float32))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)
    return loader(train_idx, True), loader(val_idx), loader(test_idx)

@torch.no_grad()
def score_model(model, loader):
    model.eval(); probabilities=[]; labels=[]
    for xb, yb in loader:
        logits = model(xb.to(DEVICE))
        probabilities.extend(torch.sigmoid(logits).cpu().numpy())
        labels.extend(yb.numpy())
    return (roc_auc_score(labels, probabilities),
            average_precision_score(labels, probabilities))

def train_model(representation, epochs=10):
    torch.manual_seed(SEED)
    model = PatientTransformer(representation).to(DEVICE)
    tr, va, te = loaders_for(representation)
    optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    positive_weight = float((y[train_idx] == 0).sum() / max(1, (y[train_idx] == 1).sum()))
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
    best_auc = -1.0; best_state = None; best_epoch = 0
    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train(); running=0.0
    
        for xb, yb in tr:
            # Hint: Reset the gradients, pass the inputs, calculate the loss.
            # Hint: Compute the gradients, update the model parameters.
    
        # Hint: Evaluate the model on the validation set, save the best model weights.
    model.load_state_dict(best_state)
    test_auc, test_auprc = score_model(model, te)
    return {'representation': representation, 'best_epoch': best_epoch,
            'validation_auroc': best_auc, 'test_auroc': test_auc,
            'test_auprc': test_auprc, 'parameters': sum(p.numel() for p in model.parameters()),
            'seconds': round(time.time()-start, 1)}

# Use ['triplet'] for a quick classroom run, or all three for comparison.
RUN_MODELS = REPRESENTATIONS
EPOCHS = int(os.environ.get('CDT_EPOCHS', '10'))
results = [train_model(name, epochs=EPOCHS) for name in RUN_MODELS]
results_df = pd.DataFrame(results)
display(results_df.round(3))

## Final Discussion

1. Which tokenization gave the highest AUROC? Was it also highest on AUPRC?
2. How might the different embedding strategies affect model performance?
3. Which strategy produces more model features? Is that always better?
4. Are we comparing only tokenization, or also the embeddings?